Daily challenge

In [ ]:
#Analyse des Accidents d'Avion et des Décès jusqu'en 2023
# 1. Importation des bibliothèques
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats


sns.set_style("whitegrid")
# 2. Chargement et nettoyage des données
# Chargement du fichier
df = pd.read_csv(
    "Airplane_Crashes_and_Fatalities_Since_1908_t0_2023.csv",
    encoding="latin1"
)

# Aperçu
print(df.head())
print(df.info())
# Conversion des dates
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

# Extraction de l'année
df["Year"] = df["Date"].dt.year

# Création des décennies
df["Decade"] = (df["Year"] // 10) * 10
# Gestion des valeurs manquantes
numeric_cols = [
    "Aboard",
    "Fatalities",
    "Ground"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Vérification des valeurs manquantes
print(df.isnull().sum())
# Création de nouvelles variables
df["Survivors"] = df["Aboard"] - df["Fatalities"]

df["SurvivalRate"] = np.where(
    df["Aboard"] > 0,
    df["Survivors"] / df["Aboard"],
    np.nan
)
# 3. Analyse Exploratoire des Données (EDA)
# Nombre total d'accidents
total_accidents = len(df)
print("Nombre total d'accidents :", total_accidents)

# Nombre total de décès
total_deaths = df["Fatalities"].sum()
print(total_deaths)


# Taux moyen de survie

print(df["SurvivalRate"].mean())
# Accidents par année
accidents_year = df.groupby("Year").size()

print(accidents_year.head())
# 4. Analyse Statistique avec SciPy
# Distribution des décès
mean_fatalities = df["Fatalities"].mean()
median_fatalities = df["Fatalities"].median()
std_fatalities = df["Fatalities"].std()

print("Moyenne :", mean_fatalities)
print("Médiane :", median_fatalities)
print("Écart-type :", std_fatalities)


"""Interprétation
La moyenne est supérieure à la médiane.
La distribution est asymétrique à droite.
Quelques catastrophes majeures augmentent fortement la moyenne."""

"""Test d'hypothèse : Avant et après 1980
Hypothèses

H0 :

Le nombre moyen de décès est identique avant et après 1980.

H1 :

Le nombre moyen de décès diffère."""

before_1980 = df[df["Year"] < 1980]["Fatalities"].dropna()

after_1980 = df[df["Year"] >= 1980]["Fatalities"].dropna()

t_stat, p_value = stats.ttest_ind(
    before_1980,
    after_1980,
    equal_var=False
)

print("t =", t_stat)
print("p =", p_value)


"""Conclusion

Comme :

p < 0.05

On rejette H0.

Le nombre moyen de décès a significativement changé entre les périodes avant et après 1980."""

# 5. Visualisations
#Évolution des accidents dans le temps
plt.figure(figsize=(14,6))

accidents_year.plot()

plt.title("Nombre d'accidents par année")
plt.xlabel("Année")
plt.ylabel("Nombre d'accidents")

plt.show()

"""Observation
Forte augmentation après la Seconde Guerre mondiale.
Pic entre les années 1940 et 1980.
Diminution progressive depuis les années 2000."""
#Décès par décennie
plt.figure(figsize=(12,6))

df.groupby("Decade")["Fatalities"].sum().plot(
    kind="bar"
)

plt.title("Décès par décennie")
plt.ylabel("Nombre de décès")

plt.show()
"""Observation

Les décennies :

1960
1970
1980

figurent parmi les plus meurtrières."""

# Histogramme des décès
plt.figure(figsize=(10,6))

sns.histplot(
    df["Fatalities"],
    bins=40,
    kde=True
)

plt.title("Distribution des décès")
plt.xlabel("Nombre de décès")

plt.show()
"""Observation
Distribution fortement asymétrique.
Beaucoup d'accidents avec peu de décès.
Quelques accidents extrêmement meurtriers.
Boxplot des décès"""
plt.figure(figsize=(8,6))

sns.boxplot(y=df["Fatalities"])

plt.title("Boxplot des décès")

plt.show()
"""Observation

Présence de nombreux outliers correspondant aux grandes catastrophes aériennes.

Taux de survie moyen par décennie"""

plt.figure(figsize=(12,6))

df.groupby("Decade")["SurvivalRate"].mean().plot(
    kind="line",
    marker="o"
)

plt.title("Taux moyen de survie par décennie")
plt.ylabel("Taux de survie")

plt.show()

Conclusion générale

L'analyse de 4 998 accidents aériens entre 1908 et 2023 révèle :

111 644 décès enregistrés.
Une moyenne de 22 décès par accident.
Une forte croissance des accidents durant l'expansion du transport aérien au XXᵉ siècle.
Une baisse progressive du nombre d'accidents récents grâce aux améliorations technologiques et réglementaires.
Une distribution très asymétrique des décès, indiquant que quelques catastrophes majeures concentrent une grande partie des pertes humaines.
Le test statistique montre une différence significative du nombre moyen de décès entre les périodes avant et après 1980.

Ces résultats suggèrent une amélioration continue de la sécurité aérienne malgré l'augmentation du trafic mondial.